# Lexical vs. semantic: does the pipeline choice matter the way we hypothesized?

**Purpose:** the motivating hypothesis behind building a second pipeline at all --
lexical suits categorical/tag-like fields, semantic suits free text -- tested head to
head using the real, shipped production code for both paths (`clustering.lexical`,
`clustering.semantic`, `clustering.kmeans`), not the investigation scaffolding from
notebooks 01-07.

**Design: a 2x2, not a single comparison.** For the same rows, cross two fields
(`tags`, categorical -- vs. `title`, free text) against both pipelines. The two
"wrong-pipeline" cells are the interesting ones, not filler.

**Scoring is deliberately different per field, not uniform:**
- **`tags`**: the field being clustered *is* the ground truth (five real, non-overlapping
  StackOverflow tags), so Adjusted Rand Index and NMI against the true groups are a
  direct, valid comparison, independent of either pipeline's own native metric
  (silhouette and c_v aren't comparable to each other -- settled in notebook 04).
- **`title`**: tag membership is an *external* signal correlated with, but not
  necessarily recoverable from, title content -- a title tagged "python" need not
  mention anything python-specific, and a genuinely good title-clustering might group by
  something tags don't capture at all. Scoring titles against tag-ARI would conflate "is
  this a good clustering of titles" with "does it happen to match an unrelated
  categorical label." Instead: c-TF-IDF top words per cluster (cheap, human-readable) and
  an LLM-as-judge pairwise comparison between the two pipelines' title-clusterings. Each
  comparison is isolated to its own call and checked once with the pipelines' letter
  assignments swapped, so a verdict that flips with the letter is distinguishable from a
  real signal.

**Also checked on diffuse data** (no known structure) to see whether any pattern found
on known structure survives when there's nothing clean to find.


In [1]:
import json
import sys
import time

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

sys.path.insert(0, "..")
sys.path.insert(0, ".")
load_dotenv("../.env")

from clustering.lexical import vectorize_lexical, find_best_kmeans_k
from clustering.semantic import vectorize_semantic, find_best_semantic_k, coherence_score
from clustering.kmeans import run_kmeans
from semantic_diagnostics import class_tfidf_top_words
from comparison_diagnostics import judge_pairwise_with_swap_check, is_available

print(f"LLM judge available: {is_available()}")


def run_lexical(texts):
    t0 = time.perf_counter()
    vect = vectorize_lexical(texts)
    k = find_best_kmeans_k(vect)
    labels = run_kmeans(vect, k)
    elapsed = time.perf_counter() - t0
    return {
        "texts": texts, "vect": vect, "labels": labels, "k": k,
        "native_metric": "silhouette", "native_score": silhouette_score(vect, labels),
        "seconds": elapsed,
    }


def run_semantic(texts):
    t0 = time.perf_counter()
    joined_texts, vect = vectorize_semantic(texts)
    k = find_best_semantic_k(joined_texts, vect)
    labels = run_kmeans(vect, k)
    elapsed = time.perf_counter() - t0
    return {
        "texts": joined_texts, "vect": vect, "labels": labels, "k": k,
        "native_metric": "c_v", "native_score": coherence_score(joined_texts, labels),
        "seconds": elapsed,
    }


def summary_row(pipeline, field, result, ari=None, nmi=None):
    row = {
        "pipeline": pipeline, "field": field, "k": result["k"],
        "native_metric": result["native_metric"], "native_score": round(result["native_score"], 4),
        "seconds": round(result["seconds"], 1),
    }
    if ari is not None:
        row["ARI"] = round(ari, 4)
        row["NMI"] = round(nmi, 4)
    return row


def top_words_table(texts, labels, top_n=8):
    texts = pd.Series(texts).reset_index(drop=True)
    labels_arr = np.asarray(labels)
    topics = class_tfidf_top_words(texts, labels_arr, top_n=top_n)
    rows = [
        {"cluster": cluster_id, "size": int((labels_arr == cluster_id).sum()), "top_words": ", ".join(words)}
        for cluster_id, words in sorted(topics.items())
    ]
    return pd.DataFrame(rows).sort_values("size", ascending=False).reset_index(drop=True)


/home/olga/Documents/personal/clustering-json-fields/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM judge available: True


## Known-structure dataset

The same construction used throughout notebooks 04-07: five real, non-overlapping
StackOverflow tags standing in for five genuinely distinct topics -- reused here as
ground truth for the `tags` field, and to pull the matching `title` field for the same
rows.

In [2]:
import random

with open("../data/stackoverflow_sample_2021_03_01_n5000.json") as f:
    data = json.load(f)

CANDIDATE_TAGS = ["python", "sql", "excel", "css", "swift"]
groups = {tag: [] for tag in CANDIDATE_TAGS}
for row in data:
    hits = [tag for tag in CANDIDATE_TAGS if tag in set(row["tags"])]
    if len(hits) == 1:
        groups[hits[0]].append(row)

# Capped and balanced the same way as notebooks/04-07's known-structure construction --
# taking every matching row unbalanced (as an earlier version of this notebook did) let
# python dominate at 65% of the data, which isn't the clean 5-group test this is meant
# to be.
rng = random.Random(42)
CAP = 90
capped = {tag: (rows if len(rows) <= CAP else rng.sample(rows, CAP)) for tag, rows in groups.items()}

rows = [row for tag in CANDIDATE_TAGS for row in capped[tag]]
true_group_labels = np.array([i for i, tag in enumerate(CANDIDATE_TAGS) for _ in capped[tag]])
known_tags = pd.Series([r["tags"] for r in rows])
known_titles = pd.Series([r["title"] for r in rows])
print(f"{len(rows)} known-structure rows across {len(CANDIDATE_TAGS)} true groups: "
      f"{[len(capped[t]) for t in CANDIDATE_TAGS]}")


418 known-structure rows across 5 true groups: [90, 90, 58, 90, 90]


## Known-structure: `tags` field (categorical)

The field being clustered *is* the ground truth here, so ARI/NMI against the true tag
groups is a direct, valid comparison.

In [3]:
lex_tags = run_lexical(known_tags)
sem_tags = run_semantic(known_tags)

tags_known_df = pd.DataFrame([
    summary_row("Lexical", "tags", lex_tags,
                adjusted_rand_score(true_group_labels, lex_tags["labels"]),
                normalized_mutual_info_score(true_group_labels, lex_tags["labels"])),
    summary_row("Semantic", "tags", sem_tags,
                adjusted_rand_score(true_group_labels, sem_tags["labels"]),
                normalized_mutual_info_score(true_group_labels, sem_tags["labels"])),
])
tags_known_df


/home/olga/Documents/personal/clustering-json-fields/venv/lib/python3.12/site-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (327) found smaller than n_clusters (330). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10842.89it/s]

,pipeline,field,k,native_metric,native_score,seconds,ARI,NMI
0,Lexical,tags,24,silhouette,0.8404,7.1,0.5314,0.7470
1,Semantic,tags,76,c_v,0.5422,21.4,0.1381,0.5581


**Is the semantic pipeline's found k here a genuine optimum, or just wherever the search
stopped?** Rather than trust a single found-k/ARI pair, sweep c_v and ARI together across
a wide range of k on the same (already-computed) embeddings -- if c_v has no real interior
peak and keeps favoring larger k indefinitely while ARI keeps getting worse, that's direct
evidence of a genuine mismatch between what c_v rewards and what recovers real categorical
structure, not a search artifact that happened to land near an arbitrary cap.

In [4]:
sweep_rows = []
for k in [5, 10, 15, 20, 30, 50, 70, 100, 122, 150, 200]:
    if k >= len(known_tags):
        continue
    labels = run_kmeans(sem_tags["vect"], k)
    sweep_rows.append({
        "k": k,
        "c_v": round(coherence_score(sem_tags["texts"], labels), 4),
        "ARI": round(adjusted_rand_score(true_group_labels, labels), 4),
    })
pd.DataFrame(sweep_rows)


,k,c_v,ARI
0,5,0.4208,0.9669
1,10,0.4150,0.7492
2,15,0.4064,0.5600
3,20,0.4367,0.3684
4,30,0.4557,0.2803
5,50,0.4759,0.1779
6,70,0.5350,0.1363
7,100,0.5704,0.0967
8,122,0.6238,0.0841
9,150,0.6894,0.0745


## Known-structure: `title` field (free text)

No ARI here (see the methodology note above) -- top words per cluster plus an LLM
pairwise judge instead.

In [5]:
lex_titles = run_lexical(known_titles)
sem_titles = run_semantic(known_titles)

titles_known_df = pd.DataFrame([
    summary_row("Lexical", "title", lex_titles),
    summary_row("Semantic", "title", sem_titles),
])
titles_known_df


,pipeline,field,k,native_metric,native_score,seconds
0,Lexical,title,4,silhouette,0.1605,8.7
1,Semantic,title,10,c_v,0.4027,14.8


In [6]:
print("=== Lexical on titles -- top words per cluster ===")
display(top_words_table(lex_titles["texts"], lex_titles["labels"]))
print("\n=== Semantic on titles -- top words per cluster ===")
display(top_words_table(sem_titles["texts"], sem_titles["labels"]))


=== Lexical on titles -- top words per cluster ===


,cluster,size,top_words
0,0,232,"python, error, sql, css, using, excel, column,..."
1,1,128,"excel, column, cell, sql, python, multiple, vb..."
2,2,36,"using, div, table, sql, data, create, css, new"
3,3,22,"app, function, using, distance, tracking, tryi..."



=== Semantic on titles -- top words per cluster ===


,cluster,size,top_words
0,9,56,"swift, ios, swiftui, view, 12, xcode, app, con..."
1,2,56,"css, react, javascript, grid, page, html, imag..."
2,0,44,"sql, column, count, sum, table, rows, tables, ..."
3,4,42,"css, div, border, height, mobile, align, navig..."
4,6,41,"python, pandas, dataframe, x, matplotlib, valu..."
5,3,40,"sql, query, column, database, snowflake, selec..."
6,5,37,"struct, s, mr, firebase, dag, calling, distanc..."
7,1,34,"cell, excel, value, values, range, based, stri..."
8,7,34,"vba, excel, sheet, value, multiple, data, send..."
9,8,34,"0, error, sum, aa, ora, 8, application, python"


In [7]:
if is_available():
    verdict_known = judge_pairwise_with_swap_check(
        "Lexical", lex_titles["texts"], lex_titles["labels"],
        "Semantic", sem_titles["texts"], sem_titles["labels"],
    )
    print(json.dumps(verdict_known, indent=2))
else:
    verdict_known = None
    print("GEMINI_API_KEY not set -- skipping LLM judge.")


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{
  "results": [
    {
      "order": "1_as_A",
      "a": "Lexical",
      "b": "Semantic",
      "coherence_a": 2,
      "coherence_b": 4,
      "winner": "Semantic",
      "reasoning": "Clustering B demonstrates significantly higher internal coherence, with several highly specific clusters like iOS/Swift development (Cluster 9), CSS/HTML layout (Cluster 4), and VBA/Excel automation (Cluster 7). In contrast, Clustering A's clusters are much broader, frequently mixing disparate topics and languages within a single cluster (e.g., Cluster 0 mixes Python, SQL, and system errors)."
    },
    {
      "order": "2_as_A",
      "a": "Semantic",
      "b": "Lexical",
      "coherence_a": 4,
      "coherence_b": 1,
      "winner": "Semantic",
      "reasoning": "Clustering A exhibits much higher internal coherence, with several clusters clearly focused on specific technologies like iOS/Swift development (Cluster 9), CSS/HTML layout (Cluster 4), or VBA/VB.NET (Cluster 7). Clustering B, however,

## Diffuse dataset

A real, diverse sample with no known group structure -- checking whether any pattern
found above survives without clean structure to find.

In [8]:
with open("../data/stackoverflow_sample_2021_03_01_n500.json") as f:
    diffuse_data = json.load(f)

diffuse_tags = pd.Series([r["tags"] for r in diffuse_data])
diffuse_titles = pd.Series([r["title"] for r in diffuse_data])
print(f"{len(diffuse_data)} diffuse rows")


500 diffuse rows


## Diffuse: `tags` field

No known groups, so no ARI/NMI -- found k, native score, and timing only, for
completeness and for the timing comparison below.

In [9]:
lex_tags_diffuse = run_lexical(diffuse_tags)
sem_tags_diffuse = run_semantic(diffuse_tags)

tags_diffuse_df = pd.DataFrame([
    summary_row("Lexical", "tags", lex_tags_diffuse),
    summary_row("Semantic", "tags", sem_tags_diffuse),
])
tags_diffuse_df


,pipeline,field,k,native_metric,native_score,seconds
0,Lexical,tags,45,silhouette,0.7191,8.5
1,Semantic,tags,62,c_v,0.4776,23.8


## Diffuse: `title` field

In [10]:
lex_titles_diffuse = run_lexical(diffuse_titles)
sem_titles_diffuse = run_semantic(diffuse_titles)

titles_diffuse_df = pd.DataFrame([
    summary_row("Lexical", "title", lex_titles_diffuse),
    summary_row("Semantic", "title", sem_titles_diffuse),
])
titles_diffuse_df


,pipeline,field,k,native_metric,native_score,seconds
0,Lexical,title,15,silhouette,0.1377,9.9
1,Semantic,title,8,c_v,0.4387,18.3


In [11]:
print("=== Lexical on diffuse titles -- top words per cluster ===")
display(top_words_table(lex_titles_diffuse["texts"], lex_titles_diffuse["labels"]))
print("\n=== Semantic on diffuse titles -- top words per cluster ===")
display(top_words_table(sem_titles_diffuse["texts"], sem_titles_diffuse["labels"]))


=== Lexical on diffuse titles -- top words per cluster ===


,cluster,size,top_words
0,2,105,"error, vs, aws, message, testing, invalid, lam..."
1,1,74,"data, column, r, js, function, error, disable,..."
2,0,42,"possible, error, use, unable, google, trying, ..."
3,14,41,"format, android, python, file, cli, end, mm, r..."
4,4,34,"number, returning, array, list, columns, value..."
5,6,32,"count, function, word, vpn, largest, library, ..."
6,7,27,"scanf, goal, clang, matplotlib, compiling, sas..."
7,11,26,"s1, s2, using, actix, punctuation, mr, differe..."
8,9,26,"working, application, mongodb, extensions, fir..."
9,5,24,"point, function, list, number, role, discord, ..."



=== Semantic on diffuse titles -- top words per cluster ===


,cluster,size,top_words
0,7,78,"react, css, js, javascript, date, input, compo..."
1,1,76,"c, function, memory, does, 1, vba, s1, s2"
2,2,72,"api, aws, android, spring, client, post, servi..."
3,4,69,"column, columns, r, sql, average, array, forma..."
4,5,68,"error, command, build, c, android, t, package,..."
5,3,60,"query, objects, object, array, combine, using,..."
6,6,40,"python, django, extract, pandas, users, vpn, w..."
7,0,37,"list, string, python, value, column, regex, ma..."


In [12]:
if is_available():
    verdict_diffuse = judge_pairwise_with_swap_check(
        "Lexical", lex_titles_diffuse["texts"], lex_titles_diffuse["labels"],
        "Semantic", sem_titles_diffuse["texts"], sem_titles_diffuse["labels"],
    )
    print(json.dumps(verdict_diffuse, indent=2))
else:
    verdict_diffuse = None
    print("GEMINI_API_KEY not set -- skipping LLM judge.")


{
  "results": [
    {
      "order": "1_as_A",
      "a": "Lexical",
      "b": "Semantic",
      "coherence_a": 2,
      "coherence_b": 4,
      "winner": "Semantic",
      "reasoning": "Clustering B is significantly more coherent as it groups items by specific technical themes, such as front-end/UI development (B7), Python-specific tasks (B6), or database operations (B3). In contrast, Clustering A's clusters often group items by superficial similarities like \"how-to\" questions or general error messages across disparate topics."
    },
    {
      "order": "2_as_A",
      "a": "Semantic",
      "b": "Lexical",
      "coherence_a": 3,
      "coherence_b": 1,
      "winner": "Semantic",
      "reasoning": "Clustering A exhibits more specific topical coherence in clusters like Cluster 6 (Python-focused issues) and Cluster 0 (data manipulation and regex), while Clustering B's clusters often group very disparate items under generic themes like 'How to...' or 'Errors:', lacking clear sub

## Observations

**On `tags` (categorical), lexical clearly wins at recovering real structure, and the
mechanism is visible, not just the outcome.** Lexical: ARI=0.531, NMI=0.747 at k=24
against the true count of 5. Semantic: ARI=0.138, NMI=0.558 at k=76 -- a real, if
partial, signal (well above a random assignment's ARI of ~0), but much weaker. The sweep
shows why: c_v rises monotonically from 0.42 at k=5 to 0.77 at k=200 with no interior
peak at all, while ARI falls monotonically from 0.97 to 0.06 over the exact same range.
This isn't a search artifact that happened to land near an arbitrary cap -- c_v's
optimization target is genuinely misaligned with recovering categorical structure here,
most likely because short, repetitive tag text gives a cluster's own c-TF-IDF words
trivially high self-coherence as clusters shrink, a sharper version of the small-cluster
reward already found and partially guarded against in `clustering/semantic.py` (see
`SEMANTIC_MIN_AVG_CLUSTER_SIZE`'s docstring in `config.py`). The diffuse `tags` result
(semantic k=62, no ground truth available there) is consistent with the same mechanism
still being in play, not a cleaner case.

**On `title` (free text), semantic clearly wins, confirmed two independent ways.** The
top-words tables make it visible directly: lexical's largest known-structure title
cluster mixes "python, error, sql, css, using, excel, column" -- all five true topics in
one cluster -- while semantic's clusters split cleanly into swift/iOS, CSS/React,
SQL/tables, CSS layout, pandas/matplotlib, and more. The LLM judge, isolated to one call
per comparison and checked with the pipelines' letter assignments swapped, agreed with
itself in both orderings on both datasets (known-structure and diffuse), citing specific
clusters each time rather than a generic preference. Lexical's own search also struggled
visibly here -- known-structure title clustering collapsed to k=4 with a low silhouette
(0.16), and the diffuse case produced two singleton clusters -- consistent with
count-vectorized free text simply not having the clean separability that count-vectorized
tags do.

**The hypothesis motivating a second pipeline holds up, in both directions, not just the
convenient one.** Categorical data was always the "easy win" to confirm; the free-text
result is the one that actually justifies the extra pipeline, and it's the more
decisively confirmed of the two here.

## Conclusion

**Lexical for categorical fields, semantic for free text -- not just a reasonable
default, but a validated choice, tested against the real shipped code for both paths
rather than assumed.** For a field like `tags`, the lexical pipeline should remain the
recommendation; for a field like `title`, semantic is the clear choice, with both
quantitative (native scores, timing) and qualitative (top words, LLM judge) evidence
pointing the same way.

**The tags sweep also confirms, with real data at this scale, the same c_v small-cluster
reward that `SEMANTIC_MIN_AVG_CLUSTER_SIZE` (see `config.py`) guards against for
smaller samples.** It's a second, independent piece of evidence for the same mechanism,
and it reinforces rather than complicates the headline result: for short, repetitive
categorical text, the lexical pipeline is the right choice outright, not a close call.
